# 第2回 宿題（発展）: 用語集の Q&A で LoRA 学習し、検索方式と比べる

演習ノートブック（`lora_format_tuning.ipynb`）と同じモデル・同じライブラリの版・同じ LoRA 設定で、FAQ の代わりに **講座の用語集（58 語）から作った Q&A** を学習させます。

比べるのは 3 つです。評価の質問はどれも同じ 28 問です。

| 方式 | 中身 |
| --- | --- |
| `before` | 学習前のモデルに質問だけを渡す |
| `rag` | 学習前のモデルに、`search_glossary`（第3回のツールと同じ検索）の結果を添えて渡す |
| `after` | LoRA で用語集を学習したモデルに質問だけを渡す |

評価の質問（`glossary_eval.json`）は 3 区分です。

| 区分 | 問数 | 正解 |
| --- | --- | --- |
| `exact`: 用語集にある語をそのまま（「LoRAとは？」） | 10 | 用語集の定義にある語句を含む |
| `paraphrase`: 言い回しを変えた質問（別名、説明から用語名を問う、など） | 10 | 同上 |
| `unknown`: 用語集にない語（「RAGとは？」） | 8 | 「用語集に載っていません」と答える（演習の `needs_review: true` に当たる） |

**学習データ**（`VARIANT` で選ぶ）

- `base`: `make_sft_dataset.py` をそのまま実行したもの。2 テンプレート × 58 語 = 116 件。
- `plus`: `make_sft_dataset.py --plus`。テンプレートを 2 つ足し（4 × 58 = 232 件）、用語集に無い語 12 個 × 2 = 24 件の「載っていません」例を加えた 256 件。

1 回の実行で 1 つの `VARIANT` を学習します。両方試すときは、**ランタイム → セッションを再起動** してから `VARIANT` を変えて上から実行します（学習後の adapter がモデルに残っているため）。

**所要時間の目安（T4）**: 準備 5 分、学習前の評価 2 つで 6〜8 分、学習 `base` 約 4 分・`plus` 約 8 分、学習後の評価 3〜4 分。

## 0. 環境の確認とライブラリの導入（演習と同じ）

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# 演習ノートブックのセクション 0 と同じ版に固定する
PINS = {"transformers": "5.18.0", "peft": "0.21.1", "trl": "1.14.2", "accelerate": "1.15.0", "datasets": "5.1.0"}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao
%pip install -q {spec}
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

`VARIANT` 以外は演習の `CONFIG` と同じ値です（`num_train_epochs`・`max_new_tokens` だけ用語集向けに変えています）。

- `num_train_epochs: 5`: `base` は 116 件で 1 エポック 15 ステップ、5 エポックで 75 ステップ。`plus` は 256 件で 32 ステップ × 5 = 160 ステップ。
- `max_new_tokens: 200`: 定義文は最長 200 字ほどあるので、演習（96）より長くしています。

In [ ]:
from pathlib import Path
import os, json, time, re

VARIANT = "base"   # "base"（make_sft_dataset.py のまま） / "plus"（--plus で作ったもの）

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 5,
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 200,
    "gen": {"do_sample": False},
    "variant": VARIANT,
    "train_file": {"base": "glossary_sft.jsonl", "plus": "glossary_sft_plus.jsonl"}[VARIANT],
    "work_dir": f"/content/glossary_lab/{VARIANT}",
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. データの読み込み

Codespaces で作った次のファイルを、このセルの「ファイル選択」でまとめてアップロードします（`/content` に置かれます）。すでに置いてあれば聞かれません。

- `glossary_sft.jsonl`（`uv run python scripts/make_sft_dataset.py > glossary_sft.jsonl`）
- `glossary_sft_plus.jsonl`（`uv run python scripts/make_sft_dataset.py --plus > glossary_sft_plus.jsonl`）。`VARIANT = "base"` だけなら無くてよい
- `glossary_eval.json`（評価の質問と正解。repo の `data/glossary_eval.json`）
- `glossary.json`（検索方式で使う。repo の `data/glossary.json`）

**見るところ**: 学習データの件数と、評価の質問が学習データと重なっていないこと（重なるとエラー）。

In [ ]:
NEEDED = [CONFIG["train_file"], "glossary_eval.json", "glossary.json"]
missing = [f for f in NEEDED if not Path(f"/content/{f}").exists()]
if missing:
    print("アップロードしてください:", missing)
    from google.colab import files
    for name in files.upload():
        print("uploaded:", name)
    # ブラウザによっては同名ファイルが「glossary_eval (1).json」のようになるので確認する
    still = [f for f in NEEDED if not Path(f"/content/{f}").exists()]
    assert not still, f"まだ足りません: {still}"

train_rows = [json.loads(l) for l in Path(f"/content/{CONFIG['train_file']}").read_text(encoding="utf-8").splitlines() if l.strip()]
EVAL = json.loads(Path("/content/glossary_eval.json").read_text(encoding="utf-8"))
GLOSSARY = json.loads(Path("/content/glossary.json").read_text(encoding="utf-8"))["terms"]
ABSTAIN = re.compile(EVAL["abstain_pattern"])
items = EVAL["items"]

train_questions = {r["messages"][0]["content"] for r in train_rows}
assert not [it["id"] for it in items if it["question"] in train_questions], "評価の質問が学習データにあります"
print(f"学習データ: {len(train_rows)} 件（{CONFIG['train_file']}）| 評価: {len(items)} 問 | 用語集: {len(GLOSSARY)} 語")
print("学習例:", train_rows[0]["messages"])
if VARIANT == "plus":
    print("『載っていません』の例:", train_rows[-1]["messages"])

## 3. モデルの読み込みと、学習前の評価（`before` と `rag`）

**採点**

- `exact` / `paraphrase`: 正解の語句（`must_include`）をすべて含み、かつ「用語集に載っていません」と逃げていなければ ○。
- `unknown`: 「用語集に載っていません／記載がありません／見つかりません」と答えれば ○（`abstain_pattern`）。何か説明してしまえば ×（**断定**）。

失敗の型は演習と同じく 2 つに分けて数えます: 用語集にある語に「載っていません」と答えた（**逃げ**）、無い語をもっともらしく説明した（**断定**）。

`rag` は、第3回の `app/tools.py` の `search_glossary` をそのまま写した関数で、`query`（質問文の中の名詞句）を検索し、結果をプロンプトに入れます。検索で見つからなければ「見つかりませんでした」という結果をそのまま渡します。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", round(model.get_memory_footprint() / 1e9, 2))

def generate(model, messages):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# ---- 第3回 app/tools.py の search_glossary と同じ処理（用語集だけアップロードしたものを使う） ----
MAX_RESULTS = 3
def _normalize(text): return text.casefold().replace(" ", "").replace("　", "")
def _names(e): return [_normalize(e["term"]), _normalize(e["id"])] + [_normalize(a) for a in e["aliases"]]
def search_glossary(query):
    q = _normalize(query)
    if not q: return "検索語が空です。調べたい用語を1つ指定してください。"
    exact = [e for e in GLOSSARY if q in _names(e)]
    partial = [e for e in GLOSSARY if e not in exact and any(q in n or (len(n) >= 3 and n in q) for n in _names(e))]
    in_def = [e for e in GLOSSARY if e not in exact and e not in partial and q in _normalize(e["definition"])]
    hits = (exact + partial + in_def)[:MAX_RESULTS]
    if not hits: return f"用語集に「{query}」は見つかりませんでした。"
    return json.dumps([{"term": e["term"], "lecture": e["lecture"], "definition": e["definition"], "related": e["related"]} for e in hits], ensure_ascii=False)

RAG_TEMPLATE = ("以下は講座の用語集を検索した結果です。\n{result}\n\n"
                "検索結果だけを根拠に、質問に日本語で簡潔に答えてください。"
                "質問の用語が検索結果に無い場合は「用語集に載っていません」と答えてください。\n\n質問: {question}")

def build_messages(item, mode):
    if mode == "rag":
        content = RAG_TEMPLATE.format(result=search_glossary(item["query"]), question=item["question"])
    else:
        content = item["question"]   # 学習データ（make_sft_dataset.py）と同じく user の質問だけ
    return [{"role": "user", "content": content}]

def score(item, raw):
    abstained = bool(ABSTAIN.search(raw))
    if item["category"] == "unknown":
        ok = abstained
        fail = "" if ok else "断定"
    else:
        has_all = all(any(w in raw for w in group) for group in item["must_include"])
        ok = has_all and not abstained
        fail = "" if ok else ("逃げ" if abstained else "語句不足")
    return ok, abstained, fail

def evaluate(model, label, mode):
    rows, t0 = [], time.time()
    for it in items:
        raw = generate(model, build_messages(it, mode))
        ok, abstained, fail = score(it, raw)
        rows.append({"id": it["id"], "category": it["category"], "question": it["question"], "raw": raw,
                     "ok": ok, "abstained": abstained, "fail": fail})
    summary = {"label": label, "n": len(rows), "seconds": round(time.time() - t0, 1)}
    for cat in ["exact", "paraphrase", "unknown"]:
        rs = [r for r in rows if r["category"] == cat]
        summary[f"{cat}_rate"] = round(sum(r["ok"] for r in rs) / len(rs), 3)
    summary["all_rate"] = round(sum(r["ok"] for r in rows) / len(rows), 3)
    summary["逃げ"] = sum(r["fail"] == "逃げ" for r in rows)
    summary["断定"] = sum(r["fail"] == "断定" for r in rows)
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)
def mark(ok): return "○" if ok else "×"

before_summary, before_rows = evaluate(model, "before", "plain")
rag_summary, rag_rows = evaluate(model, "rag", "rag")
display(pd.DataFrame([{"id": b["id"], "区分": b["category"], "質問": b["question"],
                       "before": mark(b["ok"]), "before の出力": b["raw"],
                       "rag": mark(r["ok"]), "rag の出力": r["raw"]} for b, r in zip(before_rows, rag_rows)]))

## 4. LoRA 学習

演習のセクション 4 と同じ `LoraConfig`・`SFTConfig` です。違うのは学習データ（`make_sft_dataset.py` の JSONL をそのまま `Dataset.from_list` に渡す）とエポック数だけです。

**見るところ**: `trainable%`、`Training Loss` の推移、学習時間とステップ数（`base` 75 / `plus` 160）。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(train_rows)   # 各行は {"messages": [...]}。演習の to_chat と同じ形式
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=50, save_total_limit=1,
                fp16=False, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
train_out = trainer.train()   # 途中で止まったら trainer.train(resume_from_checkpoint=True)
TRAIN_SECONDS = round(time.time() - t0)
losses = [h["loss"] for h in trainer.state.log_history if "loss" in h] or [train_out.training_loss]   # 10 ステップ未満だと途中の loss が記録されない
print(f"学習時間: {TRAIN_SECONDS} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f} | 最初の loss: {losses[0]:.3f} | 最終 loss: {losses[-1]:.3f}")
trainer.model.save_pretrained(WORK / "adapter")
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

## 5. 学習後の評価（`after`）と比較

**見るところ**

- 区分ごとの正解率が `before` → `after` でどう変わったか。`rag` と比べてどうか。
- `exact`（学習した言い方に近い）と `paraphrase`（言い回し・方向が違う）の差。特に「説明 → 用語名」を問う質問（P03・P05・P09）は、学習データとは逆向きです。
- `unknown` で **断定** が残るか。`base` の学習データには「知らない」と答える例が無く、`plus` にはあります。
- `plus` で、用語集にある語にまで「載っていません」と答える **逃げ** が出ていないか。

In [ ]:
after_summary, after_rows = evaluate(tuned, "after", "plain")
compare = pd.DataFrame([before_summary, rag_summary, after_summary]).set_index("label")[
    ["exact_rate", "paraphrase_rate", "unknown_rate", "all_rate", "逃げ", "断定", "seconds"]]
display(compare)
regressed = [b["id"] for b, a in zip(before_rows, after_rows) if b["ok"] and not a["ok"]]
print("退行（before で○ → after で×）:", regressed)
display(pd.DataFrame([{"id": b["id"], "区分": b["category"], "質問": b["question"],
                       "before": mark(b["ok"]), "rag": mark(r["ok"]), "after": mark(a["ok"]), "after の失敗の型": a["fail"],
                       "after の出力": a["raw"]} for b, r, a in zip(before_rows, rag_rows, after_rows)]))

## 6. 結果の保存

`result_<VARIANT>.json` と `eval_*.json` を `/content/glossary_lab/<VARIANT>/` に保存し、ダウンロードします。`lectures/02/` に置いて PR に含めてもかまいません。

終わったら **ランタイム → ランタイムを接続解除して削除**。もう一方の `VARIANT` も試すなら、削除の前に **セッションを再起動** して `VARIANT` を変え、上から実行します。

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k != "work_dir"},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "train_examples": len(train_rows),
    "train": {"seconds": TRAIN_SECONDS, "steps": trainer.state.global_step, "first_loss": losses[0], "final_loss": losses[-1]},
    "before": before_summary, "rag": rag_summary, "after": after_summary,
    "regressed": regressed,
    "executed_training_myself": True,
}
(WORK / f"result_{VARIANT}.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
try:
    from google.colab import files
    import zipfile
    with zipfile.ZipFile(f"/content/glossary_{VARIANT}.zip", "w") as z:   # adapter と ckpt は大きいので json だけ
        for f in WORK.glob("*.json"): z.write(f, f.name)
    files.download(f"/content/glossary_{VARIANT}.zip")
except ImportError:
    pass